In [1]:
import pandas as pd
import pickle as pkl
import os
import json
from natsort import natsorted
import re
from tqdm.auto import tqdm
from ast import literal_eval
from collections import defaultdict


In [2]:
results=os.listdir('outputs')
results=natsorted(results)

In [3]:
pkls_list=[]
for name in results:
    with open('outputs/{}'.format(name), 'rb') as f:
        pkls_list.append(pkl.load(f))

In [4]:
len(pkls_list)

13598

In [5]:
pkls_list[1000]

['{\n  "Named_Entities": {\n    "software": [\n      "passkeys",\n      "Chrome browser",\n      "Google Accounts",\n      "Google Password Manager",\n      "iCloud Keychain",\n      "Android",\n      "iOS/macOS",\n      "Windows"\n    ],\n    "hardware": [],\n    "software_vulnerabilities": [],\n    "hardware_vulnerabilities": []\n  }\n}']

In [11]:
ARTICLES_JSON = "hackerNews_updated.json"
with open(ARTICLES_JSON, "r") as f:
    data = json.load(f)

keys_in_order = list(data.keys())  # relies on insertion order; if uncertain, sort consistently
articles_text = []
for k in keys_in_order:
    fixed = data[k].get("fixed_content", [])
    articles_text.append(" ".join(fixed))

In [13]:
articles_text[2]

''

In [ ]:
import os, json, re, pickle as pkl
from time import sleep
from natsort import natsorted
from openai import OpenAI
from tqdm.auto import tqdm

# ---------- CONFIG ----------
API_KEY = ""
ARTICLES_JSON = "hackerNews_updated.json"
OUTPUTS_DIR = "outputs"  # contains 0.pkl, 1.pkl, ...
MODEL = "gpt-5-mini"
TEMPERATURE = 1.0
# ----------------------------

client = OpenAI(api_key=API_KEY)

def remove_between_tags(text):
    # Remove everything between < and > including the symbols
    text = re.sub(r'<.*?>', '', text)
    # Remove any extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text



def safe_json_loads(s: str):
    """
    Robustly parse JSON emitted by LLMs (handles code fences and stray backticks).
    """
    if not isinstance(s, str):
        raise ValueError("Expected string for JSON content.")
    s = s.strip()
    # strip code fences if present
    s = re.sub(r"^```(?:json)?\s*", "", s)
    s = re.sub(r"\s*```$", "", s)
    return json.loads(s)

def build_relation_prompt(article_text: str, entities_json: dict) -> str:
    """
    Prompt to decide relations, constrained to only the given article and entities.
    `.format`-safe (JSON braces doubled).
    """
    return """
You are a cybersecurity relation extractor. Decide ONLY from the article text provided.

Goal
- Determine whether each listed vulnerability affects any of the listed software/hardware entities.
- Consider ONLY the provided article text (no outside knowledge).
- If unsure or not stated/implied, do not create a relation.

Inputs
- Article (verbatim):
{article}

- Named entities (verbatim):
{entities}

Output format
- Return STRICT JSON with exactly this schema:

{{
  "Relations": [
    {{"vulnerability": "", "affects": "", "affects_type": "software|hardware"}}
  ]
}}

Rules
- A vulnerability name (e.g., CVE-IDs, malware/campaign names) must be in the vulnerabilities lists.
- "affects" must exactly match one entity string from software/hardware lists.
- "affects_type" = "software" if it targets software; "hardware" if it targets hardware.
- Do NOT invent entities or vulnerabilities beyond the lists.
- If no relations are supported by the article, return {{ "Relations": [] }}.
""".format(
        article=article_text.strip(),
        entities=json.dumps(entities_json, ensure_ascii=False, indent=2)
    )

def extract_relations_for_all():
    # 1) Load articles in the SAME order as your extraction run
    with open(ARTICLES_JSON, "r") as f:
        data = json.load(f)

    # keys_in_order = list(data.keys())  # relies on insertion order; if uncertain, sort consistently
    # articles_text = []
    # for k in keys_in_order:
    #     fixed = data[k].get("fixed_content", [])
    #     articles_text.append(" ".join(fixed))
    
    
    for items in tqdm(data.keys()):
        content_list=data[items]['content']
        temp=[]
        for sentences in content_list:
            fixed=remove_between_tags(sentences)
            if fixed is not None:
                temp.append(remove_between_tags(sentences))
        data[items]['fixed_content']=temp
    
    articles_text=[]
    for items in tqdm(data.keys()):
        articles_text.append(" ".join(data[items]['fixed_content']))

    # 2) Load prior entity-extraction results (0.pkl, 1.pkl, ...) in natural sort order
    output_files = natsorted(os.listdir(OUTPUTS_DIR))
    assert len(output_files) == len(articles_text), \
        f"Mismatch: {len(output_files)} outputs vs {len(articles_text)} articles."

    # 3) Build per-article relation lists (strings "Software, Vulnerability")
    software_relation_strings_per_article = []  # list[list[str]]
    hardware_relation_strings_per_article = []  # optional

    for idx, fname in enumerate(tqdm(output_files, desc="Linking relations")):
        # Load the saved LLM JSON string (you saved a list with one string)
        with open(os.path.join(OUTPUTS_DIR, fname), "rb") as f:
            loaded_obj = pkl.load(f)

        # Your earlier code stored: cnn = [result_string]; so loaded_obj is ["{...json...}"]
        if isinstance(loaded_obj, list) and loaded_obj and isinstance(loaded_obj[0], str):
            json_str = loaded_obj[0]
        elif isinstance(loaded_obj, str):
            json_str = loaded_obj
        else:
            # Fallback: skip if malformed
            software_relation_strings_per_article.append([])
            hardware_relation_strings_per_article.append([])
            continue

        # Parse the Named_Entities JSON
        try:
            entities = safe_json_loads(json_str)
        except Exception:
            # attempt to coerce single quotes -> double quotes (last resort)
            try:
                entities = json.loads(json_str.replace("'", '"'))
            except Exception:
                software_relation_strings_per_article.append([])
                hardware_relation_strings_per_article.append([])
                continue

        named = entities.get("Named_Entities", {})
        sw = named.get("software", []) or []
        hw = named.get("hardware", []) or []
        sw_vulns = named.get("software_vulnerabilities", []) or []
        hw_vulns = named.get("hardware_vulnerabilities", []) or []

        # If no vulnerabilities at all → index empty and continue (as requested)
        if len(sw_vulns) == 0 and len(hw_vulns) == 0:
            software_relation_strings_per_article.append([])
            hardware_relation_strings_per_article.append([])
            continue

        # Build prompt to decide relations (only if any vuln present)
        article_text = articles_text[idx]
        prompt = build_relation_prompt(article_text, {
            "Named_Entities": {
                "software": sw,
                "hardware": hw,
                "software_vulnerabilities": sw_vulns,
                "hardware_vulnerabilities": hw_vulns
            }
        })
        
        print(prompt)

        # Call model with low temperature for determinism
        relations_json = {"Relations": []}
        tries = 0
        while tries < 3:
            tries += 1
            try:
                resp = client.chat.completions.create(
                    model=MODEL,
                    temperature=TEMPERATURE,
                    messages=[
                        {"role": "system", "content": "You extract ONLY text-supported vulnerability→affected-entity relations."},
                        {"role": "user", "content": prompt}
                    ]
                )
                content = resp.choices[0].message.content
                print(content)
                relations_json = safe_json_loads(content)
                if "Relations" in relations_json and isinstance(relations_json["Relations"], list):
                    break
            except Exception as e:
                if "maximum context length" in str(e).lower():
                    # crude backoff: shrink article length and retry once
                    article_text_short = article_text[:8000]
                    prompt = build_relation_prompt(article_text_short, {
                        "Named_Entities": {
                            "software": sw, "hardware": hw,
                            "software_vulnerabilities": sw_vulns,
                            "hardware_vulnerabilities": hw_vulns
                        }
                    })
                sleep(1)

        # Convert to requested "comma separated entry in a list" per article
        sw_pairs = []
        hw_pairs = []

        for rel in relations_json.get("Relations", []):
            vuln = rel.get("vulnerability", "").strip()
            affects = rel.get("affects", "").strip()
            atype = rel.get("affects_type", "").strip().lower()
            if vuln and affects:
                if atype == "software":
                    sw_pairs.append(f"{affects}, {vuln}")
                elif atype == "hardware":
                    hw_pairs.append(f"{affects}, {vuln}")

        print(sw_pairs)
        print(hw_pairs)
        software_relation_strings_per_article.append(sw_pairs)
        hardware_relation_strings_per_article.append(hw_pairs)

    return software_relation_strings_per_article, hardware_relation_strings_per_article

if __name__ == "__main__":
    sw_relations, hw_relations = extract_relations_for_all()
    


    # Example: save results; you can also dump CSVs if you prefer
    with open("software_relations_per_article.pkl", "wb") as f:
        pkl.dump(sw_relations, f)
    with open("hardware_relations_per_article.pkl", "wb") as f:
        pkl.dump(hw_relations, f)

    # If you want a quick CSV for software relations:
    # Each row corresponds to an article; the cell contains a semicolon-joined list of "Software, Vulnerability"
    import csv
    with open("software_relations_per_article.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["article_index", "software_relations"])
        for i, rels in enumerate(sw_relations):
            w.writerow([i, "; ".join(rels)])


  0%|          | 0/13598 [00:00<?, ?it/s]

  0%|          | 0/13598 [00:00<?, ?it/s]

Linking relations:   0%|          | 0/13598 [00:00<?, ?it/s]


You are a cybersecurity relation extractor. Decide ONLY from the article text provided.

Goal
- Determine whether each listed vulnerability affects any of the listed software/hardware entities.
- Consider ONLY the provided article text (no outside knowledge).
- If unsure or not stated/implied, do not create a relation.

Inputs
- Article (verbatim):
The threat actors behind a new ransomware group called Hunters International have acquired the source code and infrastructure from the now-dismantled Hive operation to kick-start its own efforts in the threat landscape. "It appears that the leadership of the Hive group made the strategic decision to cease their operations and transfer their remaining assets to another group, Hunters International," Martin Zugec, technical solutions director at Bitdefender, said in a report published last week. Hive, once a prolific ransomware-as-a-service (RaaS) operation, was taken down as part of a coordinated law enforcement operation in January 2023. Wh

KeyboardInterrupt: 

In [18]:
with open('hackerNews_updated.json', 'r') as f:
    file=json.load(f)
    

for items in tqdm(file.keys()):
    content_list=file[items]['content']
    temp=[]
    for sentences in content_list:
        fixed=remove_between_tags(sentences)
        if fixed is not None:
            temp.append(remove_between_tags(sentences))
    file[items]['fixed_content']=temp

text=[]
count=0
for items in tqdm(file.keys()):
    count+=1
    text.append(file[items]['fixed_content'])

  0%|          | 0/13598 [00:00<?, ?it/s]

  0%|          | 0/13598 [00:00<?, ?it/s]

In [19]:
text[0]

['The threat actors behind a new ransomware group called Hunters International have acquired the source code and infrastructure from the now-dismantled Hive operation to kick-start its own efforts in the threat landscape.',
 '"It appears that the leadership of the Hive group made the strategic decision to cease their operations and transfer their remaining assets to another group, Hunters International," Martin Zugec, technical solutions director at Bitdefender, said in a report published last week.',
 'Hive, once a prolific ransomware-as-a-service (RaaS) operation, was taken down as part of a coordinated law enforcement operation in January 2023.',
 "While it's common for ransomware actors to regroup, rebrand, or disband their activities following such seizures, what can also happen is that the core developers can pass on the source code and other infrastructure in their possession to another threat actor.",
 'Reports about Hunters International as a possible Hive rebrand surfaced las